In [1]:
!pip install transformers datasets accelerate evaluate seqeval conllu torch -U -q



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. IMPORT LIBRARIES

In [3]:
import numpy as np
import pandas as pd
import evaluate

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    pipeline
)


# 3. LOAD POS DATASET

In [4]:
pos_dataset = load_dataset(
    "universal_dependencies",
    "en_ewt"
)

print(pos_dataset)

README.md: 0.00B [00:00, ?B/s]

C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Amrutha Reddy\.cache\huggingface\hub\datasets--universal_dependencies. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


universal_dependencies.py: 0.00B [00:00, ?B/s]

Using the latest cached version of the dataset since universal_dependencies couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'en_ewt' at C:\Users\Amrutha Reddy\.cache\huggingface\datasets\universal_dependencies\en_ewt\2.7.0\1ac001f0e8a0021f19388e810c94599f3ac13cc45d6b5b8c69f7847b2188bdf7 (last modified on Thu Jun  4 19:08:59 2026).


DatasetDict({
    train: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 12543
    })
    validation: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 2002
    })
    test: Dataset({
        features: ['idx', 'text', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc'],
        num_rows: 2077
    })
})


# 4. LOAD CHUNKING DATASET

In [5]:
chunk_dataset = load_dataset("conll2003")

print(chunk_dataset)


README.md: 0.00B [00:00, ?B/s]

C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Amrutha Reddy\.cache\huggingface\hub\datasets--conll2003. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


conll2003.py: 0.00B [00:00, ?B/s]

Using the latest cached version of the dataset since conll2003 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'conll2003' at C:\Users\Amrutha Reddy\.cache\huggingface\datasets\conll2003\conll2003\1.0.0\9a4d16a94f8674ba3466315300359b0acd891b68b6c8743ddf60b9c702adce98 (last modified on Thu Jun  4 19:09:44 2026).


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


# 5. EXTRACT POS LABELS

In [6]:
pos_labels = pos_dataset["train"].features["upos"].feature.names

print(pos_labels)

print("Number of POS Labels:", len(pos_labels))



['NOUN', 'PUNCT', 'ADP', 'NUM', 'SYM', 'SCONJ', 'ADJ', 'PART', 'DET', 'CCONJ', 'PROPN', 'PRON', 'X', '_', 'ADV', 'INTJ', 'VERB', 'AUX']
Number of POS Labels: 18


 # 6. EXTRACT CHUNK LABELS

In [7]:

chunk_labels = chunk_dataset["train"].features["chunk_tags"].feature.names

print(chunk_labels)

print("Number of Chunk Labels:", len(chunk_labels))


['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']
Number of Chunk Labels: 23


# 7. MODEL CHECKPOINT

In [8]:
model_checkpoint = "distilbert-base-uncased"


# 8. LOAD TOKENIZER

In [9]:
tokenizer = AutoTokenizer.from_pretrained(
    model_checkpoint
)


# 9. POS TOKENIZATION + LABEL ALIGNMENT

In [10]:
def tokenize_and_align_labels_pos(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["upos"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None

        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:

                label_ids.append(-100)

            elif word_idx != previous_word_idx:

                label_ids.append(label[word_idx])

            else:

                label_ids.append(label[word_idx])

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs


# 10. CHUNK TOKENIZATION + LABEL ALIGNMENT

In [11]:

def tokenize_and_align_labels_chunk(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["chunk_tags"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None

        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:

                label_ids.append(-100)

            elif word_idx != previous_word_idx:

                label_ids.append(label[word_idx])

            else:

                label_ids.append(label[word_idx])

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs



# 11. APPLY TOKENIZATION

In [12]:
tokenized_pos = pos_dataset.map(
    tokenize_and_align_labels_pos,
    batched=True
)

tokenized_chunk = chunk_dataset.map(
    tokenize_and_align_labels_chunk,
    batched=True
)


Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2002 [00:00<?, ? examples/s]

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

# 12. DATA COLLATOR

In [13]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

# 13. LOAD EVALUATION METRIC

In [14]:
metric = evaluate.load("seqeval")

# 14. COMPUTE METRICS FOR POS

In [15]:
def compute_metrics_pos(p):

    predictions, labels = p

    predictions = np.argmax(
        predictions,
        axis=2
    )

    true_predictions = [
        [
            pos_labels[p]
            for (p, l) in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [
            pos_labels[l]
            for (p, l) in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 15. COMPUTE METRICS FOR CHUNKING

In [17]:
def compute_metrics_chunk(p):

    predictions, labels = p

    predictions = np.argmax(
        predictions,
        axis=2
    )

    true_predictions = [
        [
            chunk_labels[p]
            for (p, l) in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [
            chunk_labels[l]
            for (p, l) in zip(prediction, label)
            if l != -100
        ]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }



# 16. LABEL MAPPINGS FOR POS

In [18]:
id2label_pos = {
    i: label
    for i, label in enumerate(pos_labels)
}

label2id_pos = {
    label: i
    for i, label in enumerate(pos_labels)
}

# 17. LABEL MAPPINGS FOR CHUNKING

In [19]:
id2label_chunk = {
    i: label
    for i, label in enumerate(chunk_labels)
}

label2id_chunk = {
    label: i
    for i, label in enumerate(chunk_labels)
}


# 18. LOAD POS MODEL

In [20]:
model_pos = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(pos_labels),
    id2label=id2label_pos,
    label2id=label2id_pos
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# 19. LOAD CHUNK MODEL

In [21]:
model_chunk = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(chunk_labels),
    id2label=id2label_chunk,
    label2id=label2id_chunk
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# 20. POS TRAINING ARGUMENTS

In [22]:
training_args_pos = TrainingArguments(
    output_dir="distilbert-pos",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100
)


# 21. CHUNK TRAINING ARGUMENTS

In [23]:

training_args_chunk = TrainingArguments(
    output_dir="distilbert-chunk",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100
)

# 22. CREATE POS TRAINER

In [25]:
trainer_pos = Trainer(
    model=model_pos,
    args=training_args_pos,
    train_dataset=tokenized_pos["train"],
    eval_dataset=tokenized_pos["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics_pos
)

# 23. CREATE CHUNK TRAINER

In [26]:
trainer_chunk = Trainer(
    model=model_chunk,
    args=training_args_chunk,
    train_dataset=tokenized_chunk["train"],
    eval_dataset=tokenized_chunk["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics_chunk
)

# 24. TRAIN POS MODEL

In [27]:
trainer_pos.train()

C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.167315,0.179054,0.940536,0.947994,0.944250,0.950988


C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequ

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1568, training_loss=0.30761649839732114, metrics={'train_runtime': 3703.5413, 'train_samples_per_second': 3.387, 'train_steps_per_second': 0.423, 'total_flos': 143375317432380.0, 'train_loss': 0.30761649839732114, 'epoch': 1.0})

# 25. EVALUATE POS MODEL


In [28]:
pos_results = trainer_pos.evaluate()

print(pos_results)

C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequence_labeling.py:171: UserWarning: VERB seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\sequ

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.167315,0.179054,1,0.940536,0.947994,0.944250,0.950988


{'eval_loss': 0.17905408143997192, 'eval_precision': 0.9405358399933744, 'eval_recall': 0.9479944905880879, 'eval_f1': 0.9442504365178347, 'eval_accuracy': 0.9509876822605153}


# 26. SAVE POS MODEL

In [29]:
trainer_pos.save_model(
    "saved_pos_model"
)

tokenizer.save_pretrained(
    "saved_pos_model"
)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('saved_pos_model\\tokenizer_config.json', 'saved_pos_model\\tokenizer.json')

# 27. POS INFERENCE

In [30]:
pos_classifier = pipeline(
    "token-classification",
    model="saved_pos_model",
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

sentence = "John works at Google in California"

results = pos_classifier(sentence)

print("Sentence:", sentence)

print()

for item in results:

    print(
        item["word"],
        " --> ",
        item["entity_group"]
    )


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Sentence: John works at Google in California

john  -->  PROPN
works  -->  VERB
at  -->  ADP
google  -->  PROPN
in  -->  ADP
california  -->  PROPN


# 28. TRAIN CHUNK MODEL

In [31]:
trainer_chunk.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.236610,0.232161,0.899103,0.893673,0.896380,0.943476


C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1756, training_loss=0.3425087689810299, metrics={'train_runtime': 4164.4251, 'train_samples_per_second': 3.372, 'train_steps_per_second': 0.422, 'total_flos': 148734578132280.0, 'train_loss': 0.3425087689810299, 'epoch': 1.0})

# 29. EVALUATE CHUNK MODEL

In [32]:
chunk_results = trainer_chunk.evaluate()

print(chunk_results)

C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


C:\Users\Amrutha Reddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.236610,0.232161,1,0.899103,0.893673,0.896380,0.943476


{'eval_loss': 0.23216131329536438, 'eval_precision': 0.899103139013453, 'eval_recall': 0.8936726849907949, 'eval_f1': 0.8963796873734512, 'eval_accuracy': 0.9434762578041844}


# 30. SAVE CHUNK

In [33]:
trainer_chunk.save_model(
    "saved_chunk_model"
)

tokenizer.save_pretrained(
    "saved_chunk_model"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('saved_chunk_model\\tokenizer_config.json',
 'saved_chunk_model\\tokenizer.json')

# 31. CHUNK

In [34]:
chunk_classifier = pipeline(
    "token-classification",
    model="saved_chunk_model",
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

sentence = "John works at Google in California"

results = chunk_classifier(sentence)

print("Sentence:", sentence)

print()

for item in results:

    print(
        item["word"],
        " --> ",
        item["entity_group"]
    )

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Sentence: John works at Google in California

john  -->  NP
works  -->  VP
at  -->  PP
google  -->  NP
in  -->  PP
california  -->  NP


# 32. POS TAGGING VS CHUNKING

In [35]:
print("POS Tagging vs Chunking")

print()

print("1. POS tagging identifies grammatical roles.")

print("2. Chunking identifies phrase structures.")

print("3. POS tagging is word-level classification.")

print("4. Chunking is phrase-level classification.")

print("5. Chunking is more difficult than POS tagging.")


POS Tagging vs Chunking

1. POS tagging identifies grammatical roles.
2. Chunking identifies phrase structures.
3. POS tagging is word-level classification.
4. Chunking is phrase-level classification.
5. Chunking is more difficult than POS tagging.


# 33. CHALLENGES FACED


In [36]:
print("Challenges Faced")

print()

print("1. Handling subword tokenization")

print("2. Label alignment")

print("3. Managing special tokens")

print("4. Transformer dependency compatibility")

print("5. Training large NLP models")


Challenges Faced

1. Handling subword tokenization
2. Label alignment
3. Managing special tokens
4. Transformer dependency compatibility
5. Training large NLP models


# 34. OBSERVATIONS


In [37]:
print("Observations")

print()

print("1. DistilBERT performs well for token classification.")

print("2. Proper preprocessing improves results.")

print("3. Chunking is harder than POS tagging.")

print("4. Transformers capture contextual meaning effectively.")


Observations

1. DistilBERT performs well for token classification.
2. Proper preprocessing improves results.
3. Chunking is harder than POS tagging.
4. Transformers capture contextual meaning effectively.


# 35. FINAL CONCLUSION

In [38]:
print("Conclusion")

print()

print(
    "Successfully fine-tuned DistilBERT for POS Tagging and Chunking using Hugging Face Transformers."
)

Conclusion

Successfully fine-tuned DistilBERT for POS Tagging and Chunking using Hugging Face Transformers.
